## step 1 -> Read customers table from bronze and store in df

In [0]:
df = spark.read.table("retail_data_platform.bronze_layer.sqlserver_customers")

## Step 2 -> Display the dataframe and try to identify - what cleaning can be done



In [0]:
df.display()

In [0]:
# cusotmer name should be in small letters

# cusotmer_name --> first_name and last_name then drop customer_name

# city name should be in capital letters

# postcode shouldnt contain .0

# drop the file path

# valid_from and valid_to should be in UTC

# add country column with the value 'USA'

# identify the business keys for this tabtle

## Step 3 -> Apply the required transformation steps which you identified and then again store in dataframe




In [0]:
import pyspark.sql.functions as f



df.display()

## Step 4 -> write a function in same notebook which clean and transfor the customers data from bronze and returns the dataframe



In [0]:
df = spark.read.table(
    "retail_data_platform.bronze_layer.sqlserver_customers"
)


def silver_transformations(df):
    import pyspark.sql.functions as f

    # Customer name should be lowercase
    df = (
        df.withColumn(
            "customer_name",
            f.lower("customer_name")
        )
        .withColumn(
            "customer_name",
            f.split(f.col("customer_name"), ",")
        )
        .withColumn(
            "first_name",
            f.expr("get(customer_name, 1)")
        )
        .withColumn(
            "last_name",
            f.expr("get(customer_name, 0)")
        )
        .withColumn(
            "city",
            f.upper(f.col("city"))
        )
        .withColumn(
            "postcode",
            f.regexp_replace(
                f.col("postcode"),
                r"\.0$",
                ""
            )
        )
        .withColumn(
            "valid_from",
            f.from_unixtime(
                f.expr("try_cast(valid_from as bigint)")
            )
        )
        .withColumn(
            "valid_to",
            f.from_unixtime(
                f.expr("try_cast(valid_to as bigint)")
            )
        )
        .withColumn(
            "country",
            f.lit("USA")
        )
        .drop("file_path", "customer_name")
    )

    df = df.dropDuplicates(["customer_id"])

    return df.select(
        "customer_id",
        "first_name",
        "last_name",
        "tax_id",
        "tax_code",
        "country",
        "state",
        "city",
        "postcode",
        "street",
        "number",
        "unit",
        "region",
        "district",
        "lon",
        "lat",
        "ship_to_address",
        "valid_from",
        "valid_to",
        "units_purchased",
        "loyalty_segment",
        "last_update_ts"
    )

In [0]:
df = spark.read.table("retail_data_platform.bronze_layer.sqlserver_customers")
cleaned_customers = silver_transformations(df)
cleaned_customers.display()

## Step 5 in final dataframe we will apply scd - type 1 and then store in silver schema ( before that create silver schema using SQL )


In [0]:
source_df = df


In [0]:
if spark.catalog.tableExists("retail_data_platform.bronze_layer.sqlserver_customers"):
    print("Table already exists - Now processing with SCD 1")

else:
    print("Table does not Exist - Now creating table")
    spark.sql("""Create Schema if not exists silver""")
    df.write.format("delta").mode("overwrite").saveAsTable("retail_data_platform.bronze_layer.sqlserver_customers")


    # create a common function in scd type 1
    #parametter, soruce, target ,


In [0]:
%sql
CREATE OR replace VIEW retail_customer_data_silver as
with ranked_customer as  (

    select customer_id,
    split_part(customer_name,',',2) as first_name,
    split_part(customer_name,',',1) as last_name,
    tax_id,
    tax_code,
    state,
    city,
    regexp_replace(postcode,'\\.0$','') as postcode,
    street,
    number,
    unit,
    region,
    district,
    'USA' as country,
    lon,
    lat,
    ship_to_address,
    valid_from,
    valid_to,
    units_purchased,
    loyalty_segment,
    last_update_ts,
    ROW_NUMBER() over(
        Partition by customer_id
        Order by last_update_ts DESC
    ) AS rn
    from retail_data_platform.bronze_layer.sqlserver_customers
    )

SELECT
    customer_id,
    first_name,
    last_name,
    tax_id,
    tax_code,
    state,
    city,
    postcode,
    street,
    number,
    unit,
    region,
    district,
    country,
    lon,
    lat,
    ship_to_address,
    valid_from,
    valid_to,
    units_purchased,
    loyalty_segment,
    last_update_ts
    from ranked_customer
    where rn =1




In [0]:
 %sql
select * from retail_customer_data_silver

## step 6 -> Write a function which can do scd 1 - if we provide parameters like source dataframe, business_key and target_table [ After creating this function (scd1) we will take it to common_utils and then reuse it here and in other notebooks]



## step 7 -> Fully modularize the code ( hardcoded values to json config, proper logging setup and all the resusable function should have used)